In [ ]:
# ==============================================================================
# SETUP
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, setup_logger
from datetime import datetime

# Setup logging
logger = setup_logger("incremental_facts")

# Initialize pipeline
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

logger.info("Incremental fact load initialized")

In [ ]:
# ==============================================================================
# SILVER TRANSFORMATION FUNCTIONS (Imported from shared module)
# ==============================================================================
from helpers.silver_transforms import (
    transform_service_fact,
    transform_rental_fact
)

logger.info("Silver transformation functions imported from shared module")

In [ ]:
# ==============================================================================
# FACT TABLE CONFIGURATIONS (DRY - Single Source of Truth)
# ==============================================================================

# Helper wrapper for service fact (to match TableConfig signature)
def transform_service_wrapper(service_bronze):
    """Wrapper for service fact transformation."""
    return transform_service_fact(service_bronze)

# Helper wrapper for rental fact (requires additional joins)
def transform_rental_wrapper(rental_bronze):
    """Wrapper for rental fact transformation with required joins."""
    # Load supporting tables from bronze
    staff_bronze = spark.table("wheelie.bronze.staff")
    inventory_bronze = spark.table("wheelie.bronze.inventory")
    payment_bronze = spark.table("wheelie.bronze.payment")

    return transform_rental_fact(
        rental_bronze,
        staff_bronze,
        inventory_bronze,
        payment_bronze
    )

fact_configs = [
    TableConfig(
        table_name="service",
        business_key="service_id",
        surrogate_key="service_key",
        watermark_column="service_date",
        scd_type=1,  # Facts use SCD1 logic with append mode
        gold_table_name="fact_service",
        silver_transform=transform_service_wrapper
    ),
    TableConfig(
        table_name="rental",
        business_key="rental_id",
        surrogate_key="rental_key",
        watermark_column="rental_date",
        scd_type=1,  # Facts use SCD1 logic with append mode
        gold_table_name="fact_rental",
        silver_transform=transform_rental_wrapper
    ),
]

logger.info(f"Configured {len(fact_configs)} fact tables")

In [ ]:
# ==============================================================================
# EXECUTE INCREMENTAL LOAD (DRY - Single Function Call)
# ==============================================================================

# Load all facts incrementally
results = pipeline.load_tables(fact_configs, force_full=False)

# Display results
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Fact table row counts
print("Fact Table Row Counts:")
print(f"fact_service: {spark.table('wheelie.gold.fact_service').count():,}")
print(f"fact_rental: {spark.table('wheelie.gold.fact_rental').count():,}")

# Check latest watermarks
print("\nLatest Watermarks:")
display(spark.table("wheelie.monitoring.watermarks"))